In [ ]:
%run common_deployment_config

# Pipeline Deployer

This notebook automates the deployment of Data Pipelines to Microsoft Fabric from JSON template files.

## Workflow Overview

1. **Discover Resources** - Fetch workspace ID, notebooks, and existing pipelines
2. **Scan Pipeline Files** - Find pipeline JSON templates in OneLake storage
3. **Resolve Placeholders** - Replace %%workspace_id%% and %%notebook_id%% with actual values
4. **Deploy Pipelines** - Create new pipelines with smart prefix handling (never overwrites existing ones)
5. **Report Results** - Display summary of created, skipped, and failed deployments

## Prerequisites

- Access to Microsoft Fabric workspace
- Pipeline JSON templates with placeholders in OneLake storage
- Notebooks must be deployed before pipelines (pipelines reference notebook IDs)
- Sufficient permissions to create pipelines

## Instructions

1. Update configuration parameters in Cell 5:
   - `WORKSPACE_NAME` - Workspace container name
   - `SOURCE_LAKEHOUSE` - Lakehouse containing pipeline templates
   - `BASE_FOLDER_PATH` - Base path to capability folders
   - `PIPELINES_VERSION` - Version of pipelines to deploy
2. Run Cell 11 to execute the complete pipeline deployment

## Pipeline Naming (Smart Prefix Handling)

Pipeline names are automatically prefixed based on `COMPANY_PREFIX` and `TECHNICAL_PREFIX`:
- **Source file:** `msft_claims_data_ingestion.json` (already has technical prefix)
  - Deployed as: `healthcare1_msft_claims_data_ingestion` (adds only company prefix)
- **Source file:** `my_custom_pipeline.json` (no technical prefix)
  - Deployed as: `healthcare1_msft_my_custom_pipeline` (adds both prefixes)

**Note:** Existing pipelines are never overwritten (safe by default).


---

## Import Libraries

---

## Configuration Parameters

Configure the source location for pipeline template files.

### Pipeline Source Configuration
- `WORKSPACE_NAME` - Workspace container name where pipeline templates reside
- `SOURCE_LAKEHOUSE` - Lakehouse containing pipeline template JSON files
- `BASE_FOLDER_PATH` - Base path to capability folders (e.g., "Files")
- `PIPELINES_VERSION` - Version of pipelines to deploy

> **Note:** OneLake endpoint is auto-detected from the Fabric environment. Pipeline templates are expected in folders: `{BASE_FOLDER_PATH}/{PIPELINES_VERSION}/{capability}/DataPipelines/*.json`

In [ ]:
# ============================================================================
# NOTEBOOK-SPECIFIC CONFIGURATION
# ============================================================================
# (Common config loaded from common_deployment_config)

print("✓ Pipeline deployer configuration:")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Version: {ARTIFACT_VERSION}")
print(f"  Base Dist Path: {BASE_DIST_PATH}")
fabric_client = FabricRestClient()

---

## Define Functions

All functions are defined below for pipeline discovery, placeholder resolution, and deployment.

In [ ]:
def get_workspace_id() -> str:
    """
    Get current workspace ID from common config (already auto-detected).
    Returns:
        Workspace ID (string)
    """
    print("\n🔑 Getting workspace ID...")
    workspace_id = WORKSPACE_ID
    print(f"✅ Workspace: {workspace_id}")
    return workspace_id

In [ ]:
def fetch_notebooks(workspace_id: str) -> Dict[str, str]:
    """
    Fetch all notebooks from the workspace and create name->ID mapping.
    Args:
        workspace_id: Fabric workspace ID
    Returns:
        Dictionary mapping notebook display names to IDs
    Raises:
        ValueError: If no notebooks found or API call fails
    """
    print("\n📚 Fetching notebooks from workspace...")
    
    url = f"/v1/workspaces/{workspace_id}/notebooks"
    response = fabric_client.get(url)
    response.raise_for_status()
    notebooks_data = response.json()
    notebooks = notebooks_data.get("value", [])
    notebook_map = {
        nb.get("displayName"): nb.get("id")
        for nb in notebooks
        if nb.get("displayName") and nb.get("id")
    }
    print(f"✅ Found {len(notebook_map)} notebooks")
    if not notebook_map:
        raise ValueError("❌ No notebooks found. Please deploy notebooks first.")
    return notebook_map


def fetch_existing_pipelines(workspace_id: str) -> Dict[str, str]:
    """
    Fetch all existing pipelines from the workspace and create name->ID mapping.
    Args:
        workspace_id: Fabric workspace ID
    Returns:
        Dictionary mapping pipeline display names to IDs
    """
    print("\n🔍 Fetching existing pipelines from workspace...")
    url = f"/v1/workspaces/{workspace_id}/dataPipelines"
    try:
        response = fabric_client.get(url)
        response.raise_for_status()
        pipelines_data = response.json()
        pipelines = pipelines_data.get("value", [])
        pipeline_map = {
            pl.get("displayName"): pl.get("id")
            for pl in pipelines
            if pl.get("displayName") and pl.get("id")
        }
        print(f"✅ Found {len(pipeline_map)} existing pipelines")
        return pipeline_map
    except Exception as e:
        print(f"⚠️  Could not fetch existing pipelines: {e}")
        return {}


def resolve_notebook_id(placeholder: str, notebook_map: Dict[str, str]) -> Optional[str]:
    """
    Resolve a notebook placeholder to its ID using simple pattern matching.
    
    Strategy: Given placeholder "some_note_book_name_notebook_id" and deployed name "msft_some_note_book_name",
    try matching in order:
    1. msft_some_note_book_name (with technical prefix)
    2. some_note_book_name (without prefix)
    
    Args:
        placeholder: Placeholder name (e.g., "raw_process_movement_notebook_id")
        notebook_map: Dictionary of notebook names to IDs
    Returns:
        Notebook ID if found, None otherwise
    """
    if not placeholder.endswith("_notebook_id"):
        return None
    
    # Extract base name from placeholder
    base_name = placeholder[:-len("_notebook_id")]
    
    # Pattern 1: With technical prefix (if defined)
    if TECHNICAL_PREFIX:
        pattern_with_prefix = f"{TECHNICAL_PREFIX}_{base_name}"
        for nb_name, nb_id in notebook_map.items():
            # Match if notebook name ends with the pattern or equals it (case-insensitive)
            if nb_name.lower().endswith(f"_{pattern_with_prefix.lower()}") or nb_name.lower() == pattern_with_prefix.lower():
                return nb_id
    
    # Pattern 2: Base name as-is
    for nb_name, nb_id in notebook_map.items():
        # Match if notebook name ends with the pattern or equals it (case-insensitive)
        if nb_name.lower().endswith(f"_{base_name.lower()}") or nb_name.lower() == base_name.lower():
            return nb_id
    
    # No match found
    return None

In [ ]:
def discover_pipeline_files(base_path: str) -> List[Dict[str, str]]:
    """
    Scan OneLake storage for pipeline JSON files.
    
    Args:
        base_path: ABFSS path to scan
        
    Returns:
        List of dictionaries with keys: path, name, capability
        
    Raises:
        ValueError: If no pipeline files found
    """
    # Construct pipeline-specific path from base
    base_artifacts_path: str = f"{base_path}/healthcare-artifacts/{ARTIFACT_VERSION}"
    print(f"\n📂 Scanning for pipeline files in {base_artifacts_path}...")

    pipeline_files = []
    
    for cap_entry in notebookutils.fs.ls(base_artifacts_path):
        if not cap_entry.isDir:
            continue
            
        capability_path = cap_entry.path
        capability_name = capability_path.rstrip('/').split('/')[-1]
        pipelines_dir = f"{capability_path.rstrip('/')}/DataPipelines"
        
        if not notebookutils.fs.exists(pipelines_dir):
            continue
            
        for file_entry in notebookutils.fs.ls(pipelines_dir):
            if not file_entry.isDir and file_entry.path.endswith('.json'):
                pipeline_files.append({
                    "path": file_entry.path,
                    "name": file_entry.path.split('/')[-1][:-5],  # Remove .json
                    "capability": capability_name
                })
    
    print(f"✅ Found {len(pipeline_files)} pipeline files")
    
    if not pipeline_files:
        raise ValueError("❌ No pipeline files found.")
    
    for pipeline in pipeline_files:
        print(f"  • {pipeline['name']} (from {pipeline['capability']})")
    
    return pipeline_files


def read_pipeline_content(pipeline_path: str) -> str:
    """
    Read pipeline JSON content from OneLake.
    
    Args:
        pipeline_path: ABFSS path to pipeline file
        
    Returns:
        Pipeline JSON as string
    """
    return notebookutils.fs.head(pipeline_path, 1024 * 512)

In [ ]:
def extract_placeholders(content: str) -> set:
    """
    Extract all %%placeholder%% markers from content.
    
    Args:
        content: String content to parse
        
    Returns:
        Set of placeholder names (without %% markers)
    """
    pattern = r'%%([^%]+)%%'
    return set(re.findall(pattern, content))


def resolve_placeholders(placeholders: set, workspace_id: str, notebook_map: Dict[str, str]) -> Tuple[Dict[str, str], List[str], List[Dict[str, str]]]:
    """
    Resolve all placeholders to actual values.
    
    Args:
        placeholders: Set of placeholder names
        workspace_id: Fabric workspace ID
        notebook_map: Notebook name to ID mapping
        
    Returns:
        Tuple of (resolved dict, list of missing placeholders, list of resolution details)
    """
    resolved = {}
    missing = []
    resolution_details = []
    
    for placeholder in placeholders:
        if placeholder == "workspace_id":
            resolved[placeholder] = workspace_id
            print(f"  ✅ {placeholder} → {workspace_id}")
        elif placeholder.endswith("_notebook_id"):
            notebook_id = resolve_notebook_id(placeholder, notebook_map)
            if notebook_id:
                # Find the notebook name that matched
                matched_name = next((name for name, nid in notebook_map.items() if nid == notebook_id), "unknown")
                resolved[placeholder] = notebook_id
                print(f"  ✅ {placeholder} → {matched_name} ({notebook_id[:8]}...)")
                resolution_details.append({
                    "placeholder": placeholder,
                    "notebook_name": matched_name,
                    "notebook_id": notebook_id,
                    "status": "resolved"
                })
            else:
                missing.append(placeholder)
                print(f"  ⚠️  {placeholder} → NOT FOUND")
                resolution_details.append({
                    "placeholder": placeholder,
                    "status": "not_found"
                })
        else:
            missing.append(placeholder)
            print(f"  ⚠️  {placeholder} → UNRECOGNIZED TYPE")
    
    return resolved, missing, resolution_details


def apply_replacements(content: str, resolved: Dict[str, str]) -> str:
    """
    Replace all %%placeholder%% markers with resolved values.
    
    Args:
        content: Original content with placeholders
        resolved: Dictionary of placeholder names to values
        
    Returns:
        Content with replacements applied
    """
    updated_content = content
    for placeholder, value in resolved.items():
        pattern = f"%%{placeholder}%%"
        updated_content = updated_content.replace(pattern, value)
    return updated_content

In [ ]:
def deploy_pipeline(pipeline_name: str, pipeline_json: dict, capability: str, workspace_id: str, existing_pipelines: Dict[str, str]) -> Tuple[bool, str]:
    """
    Deploy a single pipeline to Fabric workspace.
    Args:
        pipeline_name: Display name for the pipeline
        pipeline_json: Pipeline definition as dict
        capability: Capability name (for description)
        workspace_id: Target workspace ID
        existing_pipelines: Dict of existing pipeline names to IDs
    Returns:
        Tuple of (success: bool, status: str)
        Status can be: "created", "skipped_exists"
    """
    if pipeline_name in existing_pipelines:
        print(f"  ⏭️  Skipped (already exists): {pipeline_name}")
        return True, "skipped_exists"
    
    try:
        url = f"/v1/workspaces/{workspace_id}/dataPipelines"
        pipeline_json_str = json.dumps(pipeline_json)
        encoded_payload = base64.b64encode(
            pipeline_json_str.encode('utf-8')
        ).decode('utf-8')
        payload = {
            "displayName": pipeline_name,
            "description": f"Deployed from {capability}",
            "definition": {
                "parts": [{
                    "path": "pipeline-content.json",
                    "payload": encoded_payload,
                    "payloadType": "InlineBase64"
                }]
            }
        }
        response = fabric_client.post(url, json=payload)
        if response.status_code in [200, 201, 202]:
            pipeline_id = response.json().get("id", "")
            print(f"  ✅ Created: {pipeline_name} (ID: {pipeline_id})")
            return True, "created"
        else:
            print(f"  ❌ Failed ({response.status_code}): {response.text[:200]}")
            return False, "error"
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return False, "error"

print("✓ deploy_pipeline() defined")

In [ ]:
def process_single_pipeline(pipeline_info: Dict[str, str], workspace_id: str, notebook_map: Dict[str, str], existing_pipelines: Dict[str, str], unresolved_tracking: Dict[str, List[str]]) -> str:
    """
    Process and deploy a single pipeline.
    
    Args:
        pipeline_info: Dict with path, name, capability
        workspace_id: Fabric workspace ID
        notebook_map: Notebook name to ID mapping
        existing_pipelines: Dict of existing pipeline names to IDs
        unresolved_tracking: Dict to track unresolved placeholders per pipeline
    Returns:
        Status string: "created", "skipped_missing", "skipped_exists", or "error"
    """
    # Get base pipeline name from file (already has .json removed)
    base_pipeline_name = pipeline_info['name']
    
    # Apply smart prefix handling (same logic as notebooks)
    # If source already has technical prefix, add only company prefix
    # Otherwise, add both prefixes
    pipeline_name = build_artifact_name(base_pipeline_name)
    
    # Check if technical prefix already exists at the start
    tech_prefix_str = TECHNICAL_PREFIX.strip() if TECHNICAL_PREFIX else ""
    if tech_prefix_str and base_pipeline_name.startswith(f"{tech_prefix_str}_"):
        # Has tech prefix, use smart logic (add only company prefix)
        parts = []
        if COMPANY_PREFIX and COMPANY_PREFIX.strip():
            parts.append(COMPANY_PREFIX.strip())
        parts.append(base_pipeline_name)
        pipeline_name = "_".join(parts) if parts else base_pipeline_name
    
    print(f"\n--- {base_pipeline_name} → {pipeline_name} ---")
    
    try:
        content = read_pipeline_content(pipeline_info['path'])
        placeholders = extract_placeholders(content)
        
        if placeholders:
            print(f"Found {len(placeholders)} placeholder(s)")
            resolved, missing, resolution_details = resolve_placeholders(
                placeholders, workspace_id, notebook_map
            )
            
            if missing:
                print(f"  ⚠️  Skipping: Missing {len(missing)} placeholder(s)")
                # Track unresolved placeholders for this pipeline
                unresolved_tracking[pipeline_name] = missing
                return "skipped_missing"
            
            content = apply_replacements(content, resolved)
        
        pipeline_json = json.loads(content)
        
        success, status = deploy_pipeline(
            pipeline_name,
            pipeline_json,
            pipeline_info['capability'],
            workspace_id,
            existing_pipelines
        )
        
        return status if success else "error"
        
    except Exception as e:
        print(f"  ❌ Exception: {e}")
        return "error"


def process_all_pipelines(pipeline_files: List[Dict[str, str]], workspace_id: str, notebook_map: Dict[str, str], existing_pipelines: Dict[str, str]) -> Tuple[Dict[str, int], Dict[str, List[str]]]:
    """
    Process and deploy all pipelines.
    
    Args:
        pipeline_files: List of pipeline file info dicts
        workspace_id: Fabric workspace ID
        notebook_map: Notebook name to ID mapping
        existing_pipelines: Dict of existing pipeline names to IDs
    Returns:
        Tuple of (stats dict, unresolved placeholders dict)
        - stats: Dictionary with counts: created, skipped_missing, skipped_exists, error
        - unresolved: Dictionary mapping pipeline names to lists of unresolved placeholders
    """
    print(f"\n🔄 Processing {len(pipeline_files)} pipeline(s)...")
    
    stats = {
        "created": 0,
        "skipped_missing": 0,
        "skipped_exists": 0,
        "error": 0
    }
    
    unresolved_tracking = {}
    
    for pipeline_info in pipeline_files:
        status = process_single_pipeline(
            pipeline_info,
            workspace_id,
            notebook_map,
            existing_pipelines,
            unresolved_tracking
        )
        stats[status] += 1
    
    return stats, unresolved_tracking

print("✓ process_single_pipeline() and process_all_pipelines() defined")


In [ ]:
def print_summary(stats: Dict[str, int], unresolved_tracking: Dict[str, List[str]]):
    """Print execution summary with unresolved placeholders details."""
    print("\n" + "=" * 80)
    print("📊 SUMMARY")
    print("=" * 80)
    print(f"Total pipelines processed: {sum(stats.values())}")
    print(f"  ✅ Created:              {stats['created']}")
    print(f"  ⏭️  Skipped (exists):     {stats['skipped_exists']}")
    print(f"  ⚠️  Skipped (missing):    {stats['skipped_missing']}")
    print(f"  ❌ Errors:               {stats['error']}")
    
    # Print detailed unresolved placeholders
    if unresolved_tracking:
        print("\n" + "=" * 80)
        print("⚠️  UNRESOLVED PLACEHOLDERS DETAILS")
        print("=" * 80)
        for pipeline_name, missing_placeholders in sorted(unresolved_tracking.items()):
            print(f"\n📋 Pipeline: {pipeline_name}")
            for placeholder in missing_placeholders:
                print(f"  ❌ {placeholder}")
        print("\n💡 Tip: Check notebook names in workspace match the expected patterns.")
    
    print("\n✅ Deployment complete!")
    print("\nℹ️  Note: Existing pipelines are never overwritten (safe by default).")
    
    print("=" * 80)


---

## Main Orchestration Function

The main function orchestrates the complete pipeline deployment workflow.

In [ ]:
def start_pipeline_deployment() -> None:
    """
    Main execution function that orchestrates the complete pipeline deployment workflow.
    
    Steps:
        1. Display configuration summary
        2. Get workspace context
        3. Fetch notebooks and existing pipelines
        4. Discover pipeline template files
        5. Process and deploy all pipelines
        6. Display completion summary
    """
    print("\n" + "=" * 80)
    print("FABRIC PIPELINE DEPLOYER")
    print("=" * 80)
    
    # Display configuration summary
    print("\nℹ️  Configuration Summary:")
    print(f"  Workspace: {WORKSPACE_NAME}")
    print(f"\nPipeline Source:")
    print(f"  Base Path: {BASE_DIST_PATH}")
    print("=" * 80 + "\n")
    
    try:
        # Step 1: Get Workspace ID
        workspace_id = get_workspace_id()
        
        # Step 2: Fetch notebooks (once)
        notebook_map = fetch_notebooks(workspace_id)
        
        # Debug: Print available notebook names
        print("\n🔍 DEBUG: Available notebook names in workspace:")
        for i, nb_name in enumerate(sorted(notebook_map.keys()), 1):
            print(f"  {i}. {nb_name}")
        
        # Step 3: Fetch existing pipelines
        existing_pipelines = fetch_existing_pipelines(workspace_id)
        
        # Step 4: Discover pipeline files
        pipeline_files = discover_pipeline_files(BASE_DIST_PATH)
        
        # Step 5: Process all pipelines
        stats, unresolved_tracking = process_all_pipelines(
            pipeline_files,
            workspace_id,
            notebook_map,
            existing_pipelines
        )
        
        # Step 6: Report results
        print_summary(stats, unresolved_tracking)
        
    except Exception as e:
        print(f"\n❌ FATAL ERROR: {e}")
        raise

---

## Execute Pipeline Deployment

Run the main function to execute the complete pipeline deployment workflow.

**⚠️ Important:** Review configuration in Cell 5 before running this cell.

In [ ]:
start_pipeline_deployment()